# 04 — Tangent Vectors and Vector Fields

**The question:** so far we have studied *scalar* functions $f : M \to \mathbb{R}$ — one number per point. What happens when we want to attach a *direction* to each point?

## Intuition

A scalar function assigns one real number to every point of a shape. But geometry is full of quantities that are inherently directional: the direction of steepest ascent of a height function, the outward normal to a surface, the velocity of a fluid flowing along a surface.

These are **vector fields** — they attach a *vector* to each point. On a flat plane any direction in $\mathbb{R}^2$ is valid. On a curved surface there is a constraint: the vector must lie in the **tangent plane** at that point. You cannot point "through" the surface.

The shift from scalars to tangent vectors is fundamental:

| Scalar field | Tangent vector field |
|---|---|
| $f : M \to \mathbb{R}$ | $X : M \to TM$, $\; X(p) \in T_p M$ |
| one number per vertex | one 2D direction per vertex/face |
| stored as $(n,)$ array | stored as $(n, 2)$ or $(n, 3)$ array |
| e.g. temperature, height | e.g. gradient, wind velocity, normals |

## Minimal math

### The tangent plane

At every point $p$ of a smooth surface $M$ there is a **tangent plane** $T_p M$ — a flat 2D plane that best approximates $M$ near $p$. Formally:

$$T_p M = \{ \mathbf{v} \in \mathbb{R}^3 : \mathbf{v} \cdot \mathbf{n}(p) = 0 \}$$

where $\mathbf{n}(p)$ is the **outward unit normal** at $p$. The tangent plane is 2-dimensional even though it lives in $\mathbb{R}^3$.

A **tangent vector** $\mathbf{v} \in T_p M$ has three real components but satisfies one constraint ($\mathbf{v} \cdot \mathbf{n} = 0$), leaving two degrees of freedom.

### Gradient as a tangent vector field

The **gradient** $\nabla f(p)$ of a scalar function $f$ is the tangent vector that points in the direction of steepest increase of $f$ at $p$:

$$\nabla f(p) \in T_p M, \qquad \langle \nabla f(p), \mathbf{v} \rangle = \partial_{\mathbf{v}} f \quad \forall \mathbf{v} \in T_p M$$

On a triangle mesh the gradient is **constant per face** (since the function is linearly interpolated within each triangle). For a face with vertices $v_0, v_1, v_2$ and values $f_0, f_1, f_2$, setting $\mathbf{e}_{01} = v_1 - v_0$, $\mathbf{e}_{02} = v_2 - v_0$, $\mathbf{n} = \mathbf{e}_{01} \times \mathbf{e}_{02}$:

$$\nabla f = \frac{(f_1 - f_0)(\mathbf{e}_{02} \times \mathbf{n}) + (f_2 - f_0)(\mathbf{n} \times \mathbf{e}_{01})}{\|\mathbf{n}\|^2}$$

This is a 3D vector in the plane of the face. Its magnitude $\|\nabla f\|$ measures how steeply $f$ changes across the face.

In [1]:
import gsops.backend as gs
import numpy as np
import polyscope as ps

from geomfum.dataset import NotebooksDataset
from geomfum.operator import Gradient
from geomfum.shape import TriangleMesh

ps.init()

In [2]:
dataset = NotebooksDataset()
mesh = TriangleMesh.from_file(dataset.get_filename("cat-00"))

V = mesh.vertices  # (n, 3)
F = mesh.faces  # (m, 3)
n, m = mesh.n_vertices, mesh.n_faces
print(f"Cat: {n} vertices, {m} faces")

Cat: 7207 vertices, 14410 faces


## Vertex normals — the simplest vector field

The **unit normal** at each vertex is the canonical example of a tangent-perpendicular vector field. It points outward from the surface and is, by definition, perpendicular to the tangent plane.

In polyscope, `add_vector_quantity` with `defined_on="vertices"` renders one arrow per vertex.

In [3]:
normals_v = (
    mesh.vertex_normals
)  # (n, 3), area-weighted average of adjacent face normals
normals_f = mesh.face_normals  # (m, 3), one normal per face

print(f"Vertex normals shape: {normals_v.shape}")
print(f"Face   normals shape: {normals_f.shape}")
print(f"All unit length: {np.allclose(np.linalg.norm(normals_v, axis=1), 1.0):.0f}")

ps.remove_all_structures()
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)

# Vector quantities in polyscope: arrows at each vertex / face centre
ps_cat.add_vector_quantity(
    "vertex normals",
    normals_v,
    defined_on="vertices",
    enabled=True,
    radius=0.001,
    length=0.02,
    color=(0.2, 0.6, 1.0),
)
ps_cat.add_vector_quantity(
    "face normals",
    normals_f,
    defined_on="faces",
    enabled=False,
    radius=0.001,
    length=0.02,
    color=(1.0, 0.4, 0.2),
)

print(
    "Enable 'face normals' in polyscope to compare — one arrow per face vs one per vertex."
)
ps.show()

Vertex normals shape: (7207, 3)
Face   normals shape: (14410, 3)
All unit length: 1
Enable 'face normals' in polyscope to compare — one arrow per face vs one per vertex.


## The gradient: from scalar function to vector field

The gradient of $f$ is a vector field *in* the tangent plane (as opposed to normals, which are *perpendicular* to it). We compute it per-face using the formula in the math section above, then visualise both the scalar function and its gradient in polyscope.

In [4]:
def face_gradients_3d(V, F, f):
    """Per-face gradient of scalar f as 3D vectors, shape (m, 3)."""
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    f0, f1, f2 = f[F[:, 0]], f[F[:, 1]], f[F[:, 2]]

    e01 = v1 - v0
    e02 = v2 - v0
    n_vec = np.cross(e01, e02)  # face normal (unscaled), |n| = 2*area
    n2 = np.sum(n_vec**2, axis=1, keepdims=True)  # |n|²

    df1 = (f1 - f0)[:, None]
    df2 = (f2 - f0)[:, None]

    # ∇f = (df1*(e02×n) + df2*(n×e01)) / |n|²
    grad = (df1 * np.cross(e02, n_vec) + df2 * np.cross(n_vec, e01)) / n2
    return grad  # (m, 3)


f_z = V[:, 2]  # height function
grad_fz = face_gradients_3d(V, F, f_z)  # (m, 3)

# Face-centre positions (for reference)
face_centers = (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]]) / 3.0

print(f"Gradient shape (one 3D vector per face): {grad_fz.shape}")
print(f"Max gradient magnitude: {np.linalg.norm(grad_fz, axis=1).max():.4f}")

Gradient shape (one 3D vector per face): (14410, 3)
Max gradient magnitude: 1.0000


In [8]:
ps.remove_all_structures()
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)

# Scalar: height function on vertices
ps_cat.add_scalar_quantity(
    "f_z (height)", f_z, defined_on="vertices", enabled=True, cmap="reds"
)

# Vector: gradient of height, defined on faces
ps_cat.add_vector_quantity(
    "∇f_z (gradient)",
    grad_fz,
    defined_on="faces",
    enabled=True,
    radius=0.0005,
    length=0.05,
    color=(1.0, 0.3, 0.1),
)

print("Enable '∇f_z (gradient)' in polyscope — arrows point up the slope of the cat.")
print(
    "On the back (constant-z regions) the arrows are short; on steep areas they are long."
)
ps.show()

Enable '∇f_z (gradient)' in polyscope — arrows point up the slope of the cat.
On the back (constant-z regions) the arrows are short; on steep areas they are long.


## Face-defined vs vertex-defined vector fields

On a triangle mesh, geometric quantities can live on **vertices** or on **faces** (triangle centres):

| Quantity | Lives on | Shape |
|---|---|---|
| Scalar function $f$ | vertices | $(n,)$ |
| Vertex normals | vertices | $(n, 3)$ |
| Face normals | faces | $(m, 3)$ |
| Gradient of $f$ | faces (piecewise linear $f$) | $(m, 3)$ |
| Per-vertex gradient | vertices (averaged from faces) | $(n, 2)$ in local frame |

Polyscope handles both via `defined_on="vertices"` or `defined_on="faces"`. When a face quantity is needed per vertex (or vice versa), one typically averages weighted by area.

## Where to go next

- [05 — Spectral Filtering](./05_spectral_filtering.ipynb): use the eigenbasis to filter and describe shapes
- [How to compute the Laplacian?](../how_to/01_mesh_laplacian.ipynb)